## 1. Configuración e imports

# ============================================================
# ANÁLISIS EXPLORATORIO — IMPORTACIONES DE MAQUINARIA PESADA
# Fuente: Veritrade — Perú Importaciones
# Datos estructurados por: scripts/importacion_maquinaria.py
# ============================================================

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from datetime import datetime, timedelta

# Configuración
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.0f}')

print("✅ Librerías cargadas correctamente")


## 2. Cargar datos


In [ ]:
# Buscar el archivo estructurado más reciente
outputs_dir = Path('../outputs')
archivos = sorted(outputs_dir.glob('*_estructurado.xlsx'))

if not archivos:
    raise FileNotFoundError("❌ No hay archivos _estructurado.xlsx en outputs/")

archivo = archivos[-1]  # El más reciente
print(f"📂 Cargando: {archivo.name}")

df = pd.read_excel(archivo, sheet_name='estructurado')

print(f"✅ {len(df):,} registros cargados")
print(f"✅ {len(df.columns)} columnas")
print(f"✅ Período: {df['fecha_dua'].min()} a {df['fecha_dua'].max()}")


## 3. KPIs principales


In [ ]:
# Calcular indicadores clave
total_unidades = len(df)
total_cif = df['cif_usd'].sum()
marcas_unicas = df['marca'].nunique()
modelos_unicos = df['modelo'].nunique()
precio_promedio = df['cif_usd'].mean()

# Mostrar tarjetas
print("=" * 60)
print("📊 INDICADORES CLAVE (KPIs)")
print("=" * 60)
print(f"   Unidades importadas:  {total_unidades:>8,}")
print(f"   Valor CIF total:      ${total_cif:>12,.0f}")
print(f"   Marcas distintas:     {marcas_unicas:>8}")
print(f"   Modelos distintos:    {modelos_unicos:>8}")
print(f"   Precio promedio:      ${precio_promedio:>10,.0f}")
print("=" * 60)

###  4. Distribución por tipo de maquinaria


In [ ]:
# Distribución por categoría
cat_counts = df['categoria_maquinaria'].value_counts().head(10)

fig = px.bar(
    x=cat_counts.values,
    y=cat_counts.index,
    orientation='h',
    title='🏗️ Distribución por Tipo de Maquinaria',
    labels={'x': 'Unidades', 'y': 'Categoría'},
    text=cat_counts.values
)
fig.update_traces(textposition='outside')
fig.update_layout(height=500)
fig.show()

## 5.  Market Share por Grupo Importador


In [ ]:
# Participación de mercado por grupo importador
grupo_share = (
    df.groupby('grupo_importador')
    .agg(Unidades=('dua_dam', 'count'), CIF_USD=('cif_usd', 'sum'))
    .sort_values('Unidades', ascending=False)
    .head(10)
)

grupo_share['Share%'] = (grupo_share['Unidades'] / total_unidades * 100).round(1)

print("🏆 TOP 10 GRUPOS IMPORTADORES")
print("=" * 70)
for i, (grupo, row) in enumerate(grupo_share.iterrows(), 1):
    bar = '█' * int(row['Share%'])
    print(f"{i:2}. {grupo:<30} {row['Unidades']:>5} unidades  ({row['Share%']:>5.1f}%)  {bar}")


## 6. Top 10 Marcas

In [ ]:
por_mes = df.set_index("fecha").resample("MS").size()
ax = por_mes.plot(marker="o", color="#4C78A8")
ax.set_title("Unidades importadas por mes"); ax.set_ylabel("unidades"); ax.set_xlabel("")
plt.tight_layout(); plt.show()
df["fecha"].dt.year.value_counts().sort_index().to_frame("unidades")


## 7. Distribución por Tren de Rodaje


In [ ]:
# Orugas vs Ruedas
tren_counts = df['tren_rodaje'].value_counts()

fig = px.pie(
    values=tren_counts.values,
    names=tren_counts.index,
    title='🔩 Distribución por Tren de Rodaje',
    color_discrete_sequence=['#8B4513', '#4169E1']
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()


## 8.  Tendencia mensual


In [ ]:
# Importaciones por mes
df['fecha_dua'] = pd.to_datetime(df['fecha_dua'])
df['mes_anio'] = df['fecha_dua'].dt.to_period('M').astype(str)

mensual = df.groupby('mes_anio').agg(
    Unidades=('dua_dam', 'count'),
    CIF_USD=('cif_usd', 'sum')
).tail(24)  # Últimos 24 meses

fig = go.Figure()
fig.add_trace(go.Bar(
    x=mensual.index,
    y=mensual['Unidades'],
    name='Unidades',
    marker_color='#1F4E79'
))
fig.add_trace(go.Scatter(
    x=mensual.index,
    y=mensual['CIF_USD'],
    name='CIF USD',
    yaxis='y2',
    line=dict(color='#C0392B', width=2)
))
fig.update_layout(
    title='📈 Tendencia Mensual de Importaciones',
    xaxis_title='Mes',
    yaxis_title='Unidades',
    yaxis2=dict(title='CIF USD', overlaying='y', side='right'),
    height=500
)
fig.show()


## 9.  Distribución por Peso


In [ ]:
# Categorías de peso
peso_order = ['-14t', '14-17t', '17t-21t', '21-27t', '27t-33t', '33t-38t', '38-50t', '50t+', '100t+']
peso_counts = df['categoria_peso'].value_counts().reindex(peso_order).dropna()

fig = px.bar(
    x=peso_counts.index,
    y=peso_counts.values,
    title='⚖️ Distribución por Categoría de Peso',
    labels={'x': 'Rango de Peso', 'y': 'Unidades'},
    text=peso_counts.values
)
fig.update_traces(textposition='outside', marker_color='#2E86C1')
fig.show()


### 10. Resumen de calidad


In [ ]:
# Métricas de calidad
print("📊 CALIDAD DE DATOS")
print("=" * 50)

calidad = {
    'Confianza ALTA': (df['confianza_clasificacion'] == 'ALTA').sum(),
    'Confianza MEDIA': (df['confianza_clasificacion'] == 'MEDIA').sum(),
    'Confianza BAJA': (df['confianza_clasificacion'] == 'BAJA').sum(),
    'Marca detectada': df['marca'].notna().sum(),
    'Modelo detectado': df['modelo'].notna().sum(),
    'Tren rodaje detectado': df['tren_rodaje'].notna().sum(),
}

for k, v in calidad.items():
    pct = v / total_unidades * 100
    bar = '█' * int(pct / 2)
    print(f"  {k:<25} {v:>6} ({pct:>5.1f}%)  {bar}")


## 11. Exportar resumen


In [ ]:
# Guardar resumen en Excel
resumen_path = outputs_dir / f"{archivo.stem}_resumen_eda.xlsx"

with pd.ExcelWriter(resumen_path, engine='openpyxl') as writer:
    grupo_share.to_excel(writer, sheet_name='Grupos_Importador')
    marca_share.to_frame('Unidades').to_excel(writer, sheet_name='Top_Marcas')
    mensual.to_excel(writer, sheet_name='Tendencia_Mensual')
    df['categoria_maquinaria'].value_counts().to_frame('Unidades').to_excel(writer, sheet_name='Categorias')

print(f"✅ Resumen guardado en: {resumen_path}")
print("🚀 Análisis completado.")


## Cierre y próximos pasos

- Tabla apta para EDA de mercado: volumen, marcas, importadores, origen, valor CIF y specs.
- **Limitaciones**: `modelo`/`version` incompletos (~21% formato sucio) y `largo_mm` con unidades mezcladas.
- Si `modelo`/`version` resultan clave, evaluar la *fase B* (LLM solo para ese ~21%).
